# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library and reference data components by their `@id` fields, following best practices for programmatic reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nIdentifier:", getattr(metadata, 'identifier', None))
print("Published:", getattr(metadata, 'datePublished', None))
print("Keywords:", getattr(metadata, 'keywords', None))

## 2. Data Overview

Review available record sets and fields. All components are referenced by their `@id`. Let's examine which RecordSets are available and sample their fields.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)
print("Available RecordSets (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")

# Show the fields for each record set
print("\nRecordSet fields overview:")
record_set_to_fields = {}
for rs in record_sets:
    fields = dataset.record_set_schema(rs['@id']).fields
    record_set_to_fields[rs['@id']] = [field['@id'] for field in fields]
    print(f"RecordSet {rs['@id']}: Fields {[field['@id'] for field in fields]}")

## 3. Data Extraction

We load data from each record set into a DataFrame for analysis, referencing record sets and field columns exclusively by their `@id`.

Below, all dataframes extracted will be stored in a dictionary, with keys as record set `@id`.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for record_set in record_sets:
    rs_id = record_set['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"Warning: RecordSet {rs_id} is empty.")

# List available DataFrames
if dataframes:
    print("Loaded DataFrames for record sets (by @id):", list(dataframes.keys()))
    df_example_key = list(dataframes.keys())[0]
    print(f"\nSample columns for {df_example_key}:\n", dataframes[df_example_key].columns.tolist())
    dataframes[df_example_key].head()
else:
    print("No tabular data could be loaded.")

## 4. Exploratory Data Analysis (EDA)

We demonstrate classic EDA steps. Please replace IDs with actual ones from the above cells if needed. We'll select a numeric field by its `@id`, filter high values, normalize, and optionally group by a field.

In [ ]:
# Choose a record set and numeric field by @id for analysis
if dataframes:
    # Use the first record set as an example
    target_rs_id = list(dataframes.keys())[0]
    df = dataframes[target_rs_id]

    # Heuristically pick a numeric field by column dtype
    numeric_columns = df.select_dtypes(include=['float', 'int']).columns
    if len(numeric_columns) > 0:
        numeric_field_id = numeric_columns[0]
        print(f"Numeric field selected (by @id): {numeric_field_id}")

        # Set a threshold for filtering
        threshold = float(df[numeric_field_id].mean()) if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-6)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another (non-numeric) column
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < len(df)/2 and df[col].dtype=='object']
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric field for grouping found.")
    else:
        print("No numeric fields detected for EDA in this record set.")
else:
    print("No available data for EDA section.")

## 5. Visualization

Below, visualize the distribution of the numeric field used above and (if grouped) the aggregate means by group field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'group_field_id' in locals():
        plt.figure(figsize=(8,4))
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
        plt.title(f'Average {numeric_field_id} by {group_field_id}')
        plt.ylabel(f'Average {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric fields found to visualize.')

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to load a Croissant-specified dataset, identify record sets and fields by their `@id`, extract tabular data, filter and normalize numeric fields, and visualize distributions and group effects. For further insights, explore more columns and record sets as needed.

Please consult the FAIR^2 dataset schema or contact the authors for detailed field definitions and metadata descriptions.